# 🌫️ 미세먼지 × 건강영향(호흡기·심혈관질환) 분석
**데이터**: Air Quality and Health Impact Dataset (Kaggle)

---
### ⚙️ 사전 준비 (최초 1회만)
1. [Kaggle 계정] → Settings → API → **Create New Token** → `kaggle.json` 다운로드
2. 아래 STEP 0 셀 실행 후 `kaggle.json` 업로드

## STEP 0. Kaggle 인증

In [ ]:
# kaggle.json 업로드
from google.colab import files
files.upload()  # kaggle.json 선택

import os, shutil
os.makedirs(os.path.expanduser('~/.kaggle'), exist_ok=True)
shutil.copy('kaggle.json', os.path.expanduser('~/.kaggle/kaggle.json'))
os.chmod(os.path.expanduser('~/.kaggle/kaggle.json'), 0o600)
print('✅ Kaggle 인증 완료')

## STEP 1. 데이터 다운로드

In [ ]:
!pip install -q kaggle

# 메인 데이터셋: PM2.5·PM10·AQI·NO2·SO2·O3 + HealthImpactClass 포함
!kaggle datasets download -d rabieelkharoua/air-quality-and-health-impact-dataset --unzip -q

import glob, os
csvs = glob.glob('*.csv')
print('다운로드된 파일:', csvs)

## STEP 2. 데이터 로드 및 컬럼 확인

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# 한글 폰트
import subprocess
subprocess.run(['apt-get','-qq','install','fonts-nanum'], check=False)
fm.fontManager.addfont('/usr/share/fonts/truetype/nanum/NanumGothic.ttf')
plt.rcParams.update({'font.family':'NanumGothic','axes.unicode_minus':False,'figure.dpi':110})

# CSV 로드 (파일명 자동 탐색)
csv_path = csvs[0]
df_raw = pd.read_csv(csv_path)
print(f'크기: {df_raw.shape}')
print('컬럼:', df_raw.columns.tolist())
df_raw.head(3)

## STEP 3. 전처리

In [ ]:
df = df_raw.copy()
df.columns = [c.strip().lower().replace(' ','_') for c in df.columns]

# ── 독립변수 컬럼 매핑 (대소문자·표기 차이 자동 대응) ──
COL_MAP = {
    'pm10'       : [c for c in df.columns if 'pm10' in c],
    'pm2_5'      : [c for c in df.columns if 'pm2' in c and '5' in c],
    'aqi'        : [c for c in df.columns if 'aqi' in c],
    'no2'        : [c for c in df.columns if 'no2' in c],
    'so2'        : [c for c in df.columns if 'so2' in c],
    'o3'         : [c for c in df.columns if c.startswith('o3')],
    'temperature': [c for c in df.columns if 'temp' in c],
    'humidity'   : [c for c in df.columns if 'humid' in c],
    'wind_speed' : [c for c in df.columns if 'wind' in c],
    'health_class': [c for c in df.columns if 'health' in c or 'impact' in c or 'class' in c],
    'respiratory' : [c for c in df.columns if 'resp' in c],
    'cardiovascular': [c for c in df.columns if 'cardio' in c or 'cardiov' in c],
}

# 첫 번째 매칭 컬럼만 선택
COLS = {k: v[0] for k, v in COL_MAP.items() if v}
print('사용할 컬럼 매핑:')
for k, v in COLS.items():
    print(f'  {k:15s} → {v}')

In [ ]:
# 분석 컬럼만 추출
use_cols = list(set(COLS.values()))
df = df[use_cols].copy()

print('=== 결측치 현황 ===')
print(df.isnull().sum())

# 결측치 처리
before = len(df)
num_cols = df.select_dtypes(include='number').columns
df[num_cols] = df[num_cols].fillna(df[num_cols].mean())   # 연속형: 평균 대체
df = df.dropna()                                           # 범주형 결측: 제거
print(f'\n결측치 처리 후: {before}→{len(df)}행')

# 이상치 처리 (연속형 변수, IQR)
for col in [COLS.get('pm10'), COLS.get('pm2_5'), COLS.get('aqi')]:
    if col and col in df.columns:
        Q1, Q3 = df[col].quantile([0.25, 0.75])
        IQR = Q3 - Q1
        mask = (df[col] >= Q1 - 1.5*IQR) & (df[col] <= Q3 + 1.5*IQR)
        removed = (~mask).sum()
        df = df[mask]
        if removed: print(f'이상치 제거 [{col}]: {removed}건')

print(f'\n✅ 전처리 완료: {len(df):,}행')
df.describe().round(2)

## STEP 4. 기술통계

In [ ]:
# 독립변수 기술통계
indep_cols = [v for k, v in COLS.items() 
              if k in ['pm10','pm2_5','aqi','no2','so2','o3','temperature','humidity','wind_speed']]
indep_cols = [c for c in indep_cols if c in df.columns]

stat = df[indep_cols].describe().round(2)
stat.index = ['N','평균','표준편차','최솟값','Q1','중앙값','Q3','최댓값']
print('=== 독립변수 기술통계 ==='); print(stat.to_string())

# 종속변수 분포
hc = COLS.get('health_class')
if hc and hc in df.columns:
    print(f'\n=== {hc} 분포 ===')
    print(df[hc].value_counts().to_string())

## STEP 5. 시각화

In [ ]:
# ── 그래프 1: PM10 · PM2.5 분포 비교 ────────────────────
pm10_col = COLS.get('pm10')
pm25_col = COLS.get('pm2_5')

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
fig.suptitle('미세먼지 농도 분포', fontsize=14, fontweight='bold')

for ax, col, label, color in zip(
    axes,
    [pm10_col, pm25_col],
    ['PM10 (μg/m³)', 'PM2.5 (μg/m³)'],
    ['#4A90D9', '#E74C3C']
):
    if col and col in df.columns:
        ax.hist(df[col], bins=30, color=color, edgecolor='white', linewidth=0.6)
        ax.axvline(df[col].mean(), color='black', ls='--', lw=1.5,
                   label=f'평균 {df[col].mean():.1f}')
        ax.set_xlabel(label); ax.set_ylabel('빈도')
        ax.set_title(label); ax.legend()

plt.tight_layout()
plt.savefig('plot1_pm_distribution.png', bbox_inches='tight')
plt.show()

In [ ]:
# ── 그래프 2: 건강영향 등급별 PM10·PM2.5 박스플롯 ────────
hc = COLS.get('health_class')

if hc and hc in df.columns:
    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    fig.suptitle('건강영향 등급별 미세먼지 농도 분포', fontsize=14, fontweight='bold')

    palette = 'RdYlGn_r'
    for ax, col, label in zip(axes,
                              [pm10_col, pm25_col],
                              ['PM10 (μg/m³)', 'PM2.5 (μg/m³)']):
        if col and col in df.columns:
            order = sorted(df[hc].unique())
            sns.boxplot(data=df, x=hc, y=col, order=order,
                        palette=palette, ax=ax, linewidth=0.8,
                        flierprops=dict(marker='o', markersize=3, alpha=0.4))
            ax.set_xlabel('건강영향 등급'); ax.set_ylabel(label)
            ax.set_title(label)
            ax.tick_params(axis='x', rotation=15)

    plt.tight_layout()
    plt.savefig('plot2_boxplot_by_class.png', bbox_inches='tight')
    plt.show()
else:
    print('⚠ health_class 컬럼 없음 — 건너뜀')

In [ ]:
# ── 그래프 3: 상관관계 히트맵 ─────────────────────────────
fig, ax = plt.subplots(figsize=(9, 7))

num_df = df.select_dtypes(include='number')
corr = num_df.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))

sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, vmin=-1, vmax=1,
            mask=mask, linewidths=0.4, square=True,
            annot_kws={'size': 9}, ax=ax)
ax.set_title('변수 간 상관관계 히트맵', fontsize=13, fontweight='bold')
ax.set_xticklabels(ax.get_xticklabels(), rotation=35, ha='right')

plt.tight_layout()
plt.savefig('plot3_heatmap.png', bbox_inches='tight')
plt.show()

In [ ]:
# ── 그래프 4: PM2.5 × AQI 산점도 + 건강 등급 색상 ─────────
aqi_col = COLS.get('aqi')
hc      = COLS.get('health_class')

if pm25_col and aqi_col and pm25_col in df.columns and aqi_col in df.columns:
    fig, ax = plt.subplots(figsize=(8, 6))

    if hc and hc in df.columns:
        classes = sorted(df[hc].unique())
        palette = plt.cm.get_cmap('RdYlGn_r', len(classes))
        for i, cls in enumerate(classes):
            sub = df[df[hc] == cls]
            ax.scatter(sub[pm25_col], sub[aqi_col],
                       c=[palette(i)], s=20, alpha=0.55,
                       label=str(cls), edgecolors='none')
        ax.legend(title='건강영향 등급', fontsize=8, title_fontsize=9)
    else:
        ax.scatter(df[pm25_col], df[aqi_col], s=15, alpha=0.4, color='#4A90D9')

    # 회귀선
    sl, ic, r, p, _ = stats.linregress(df[pm25_col], df[aqi_col])
    xr = np.linspace(df[pm25_col].min(), df[pm25_col].max(), 200)
    ax.plot(xr, sl*xr+ic, color='black', lw=1.8,
            label=f'회귀선  r={r:.3f}  p={p:.3f}')

    ax.set_xlabel('PM2.5 (μg/m³)'); ax.set_ylabel('AQI')
    ax.set_title('PM2.5 × AQI 산점도 (건강영향 등급 색상)', fontsize=12, fontweight='bold')
    ax.legend(fontsize=8)

    plt.tight_layout()
    plt.savefig('plot4_scatter_pm25_aqi.png', bbox_inches='tight')
    plt.show()
    print(f'\n📌 PM2.5 × AQI 상관계수 r={r:.4f}  p={p:.4f}')
else:
    print('⚠ PM2.5 또는 AQI 컬럼 없음')

In [ ]:
# ── 그래프 5: 호흡기·심혈관 사례 수 (있을 경우) ───────────
resp_col  = COLS.get('respiratory')
cardio_col = COLS.get('cardiovascular')
present = [(c, l) for c, l in [(resp_col,'호흡기'), (cardio_col,'심혈관')] if c and c in df.columns]

if present:
    fig, axes = plt.subplots(1, len(present), figsize=(6*len(present), 4))
    if len(present) == 1: axes = [axes]
    fig.suptitle('질환 발생 사례 분포', fontsize=13, fontweight='bold')

    colors = ['#E67E22', '#8E44AD']
    for ax, (col, label), color in zip(axes, present, colors):
        if df[col].nunique() <= 15:   # 범주형
            vc = df[col].value_counts().sort_index()
            ax.bar(vc.index.astype(str), vc.values, color=color, edgecolor='white')
            ax.set_xlabel(label)
        else:                          # 연속형
            ax.hist(df[col], bins=25, color=color, edgecolor='white')
            ax.set_xlabel(label)
        ax.set_ylabel('빈도'); ax.set_title(f'{label} 사례 분포')

    plt.tight_layout()
    plt.savefig('plot5_disease_cases.png', bbox_inches='tight')
    plt.show()
else:
    print('ℹ 호흡기·심혈관 사례 컬럼 없음 — 건너뜀')

## STEP 6. 통계 검정 — PM 농도와 건강등급 관계

In [ ]:
hc = COLS.get('health_class')

for pm_col, pm_label in [(pm10_col, 'PM10'), (pm25_col, 'PM2.5')]:
    if not (pm_col and hc and pm_col in df.columns and hc in df.columns):
        continue

    groups = [grp[pm_col].dropna().values for _, grp in df.groupby(hc)]
    if len(groups) >= 2:
        f_stat, p_val = stats.f_oneway(*groups)
        sig = '★ 유의미 (p<0.05)' if p_val < 0.05 else '비유의 (p≥0.05)'
        print(f'{pm_label} × 건강등급  →  F={f_stat:.3f}  p={p_val:.4f}  {sig}')

# PM10 vs PM2.5 상관
if pm10_col and pm25_col and pm10_col in df.columns and pm25_col in df.columns:
    r, p = stats.pearsonr(df[pm10_col].dropna(), df[pm25_col].dropna())
    print(f'\nPM10 × PM2.5 상관계수  r={r:.4f}  p={p:.4f}')

## STEP 7. 결과 저장

In [ ]:
df.to_csv('분석데이터_최종.csv', index=False, encoding='utf-8-sig')

files_saved = ['분석데이터_최종.csv',
               'plot1_pm_distribution.png', 'plot2_boxplot_by_class.png',
               'plot3_heatmap.png', 'plot4_scatter_pm25_aqi.png',
               'plot5_disease_cases.png']

print('저장 완료:')
for f in files_saved:
    if os.path.exists(f):
        print(f'  ✔ {f}  ({os.path.getsize(f):,} bytes)')

print('\n✅ 모든 분석 완료!')

---
### 📋 결과 요약
| 파일 | 내용 |
|------|------|
| plot1 | PM10·PM2.5 농도 분포 히스토그램 |
| plot2 | 건강영향 등급별 PM 농도 박스플롯 |
| plot3 | 전체 변수 상관관계 히트맵 |
| plot4 | PM2.5 × AQI 산점도 + 회귀선 |
| plot5 | 호흡기·심혈관 사례 분포 |
| STEP 6 | 일원분산분석(ANOVA) + 피어슨 상관 |

**다음 단계**: 로지스틱 회귀 / 랜덤포레스트 예측 모델로 확장 가능